In [1]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from rapidfuzz import process, fuzz
import re
import unicodedata
from utils.fetch_data import fetch_soil, fetch_weather

In [2]:
def normalize_text(x):
    x = str(x).lower().strip()
    x = unicodedata.normalize("NFKD", x)
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x


def fuzzy_match_district(df, centroid, threshold=85):
    # Normalize both sides
    df["District_norm"] = df["District"].apply(normalize_text)
    centroid["District_norm"] = centroid["District"].apply(normalize_text)
    centroid["State_norm"] = centroid["State"].apply(normalize_text)
    df["State_norm"] = df["State"].apply(normalize_text)

    # Build fast lookup per state
    state_to_districts = (
        centroid.groupby("State_norm")["District_norm"].apply(list).to_dict()
    )

    def match_row(row):
        dist = row["District_norm"]
        state = row["State_norm"]

        if state in state_to_districts and dist in state_to_districts[state]:
            return dist

        # Try fuzzy match in same state first
        if state in state_to_districts:
            match, score, _ = process.extractOne(
                dist, state_to_districts[state], scorer=fuzz.WRatio
            )  # type: ignore
            if score >= threshold:
                return match

        # If still not found, fuzzy match across all districts
        all_dists = centroid["District_norm"].tolist()
        match, score, _ = process.extractOne(dist, all_dists, scorer=fuzz.WRatio)  # type: ignore
        if score >= threshold:
            return match

        return dist

    df["District_norm"] = df.apply(match_row, axis=1)

    merged = df.merge(
        centroid[["State_norm", "District_norm", "Latitude", "Longitude"]].rename(
            columns={
                "Latitude": "latitude",
                "Longitude": "longitude",
            }
        ),
        on=["State_norm", "District_norm"],
        how="left",
    )

    return merged


In [3]:
df = pd.read_csv("data/converted_crop_data.csv")
centroid = pd.read_csv("data/centroid.csv")
df["Start_Year"] = df["Year"].apply(lambda x: int(x.split("-")[0]))

df_merged = fuzzy_match_district(df, centroid)
df_merged.dropna(subset=["latitude", "longitude"], inplace=True)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 75714 entries, 0 to 79426
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State           75714 non-null  object 
 1   District        75714 non-null  object 
 2   Season          75714 non-null  object 
 3   Year            75714 non-null  object 
 4   Crop            75714 non-null  object 
 5   Area_Ha         75714 non-null  float64
 6   Production_Ton  75215 non-null  float64
 7   Yield_TonHa     75714 non-null  float64
 8   Start_Year      75714 non-null  int64  
 9   District_norm   75714 non-null  object 
 10  State_norm      75714 non-null  object 
 11  latitude        75714 non-null  float64
 12  longitude       75714 non-null  float64
dtypes: float64(5), int64(1), object(7)
memory usage: 8.1+ MB


In [4]:
def make_batches(df, batch_size=50):
    return np.array_split(df, max(1, len(df) // batch_size))


def append_data(df_merged):
    soil_keys = df_merged[
        ["State", "District", "latitude", "longitude"]
    ].drop_duplicates()
    soil_batches = make_batches(soil_keys, batch_size=50)

    soil_records = []
    print(f"Fetching soil data in {len(soil_batches)} batches...")

    for batch in tqdm(soil_batches):
        for _, row in batch.iterrows():
            data = fetch_soil(row["latitude"], row["longitude"])
            data["State"] = row["State"]
            data["District"] = row["District"]
            soil_records.append(data)

        time.sleep(2)  # polite delay between batches

    soil_df = pd.DataFrame(soil_records)
    df_merged = df_merged.merge(soil_df, on=["State", "District"], how="left")

    weather_keys = df_merged[
        ["State", "District", "latitude", "longitude", "Season", "Start_Year"]
    ].drop_duplicates()

    weather_batches = make_batches(weather_keys, batch_size=50)

    weather_records = []
    print(f"Fetching weather data in {len(weather_batches)} batches...")

    for batch in tqdm(weather_batches):
        for _, row in batch.iterrows():
            data = fetch_weather(
                row["latitude"], row["longitude"], row["Start_Year"], row["Season"]
            )
            data["State"] = row["State"]
            data["District"] = row["District"]
            data["Season"] = row["Season"]
            data["Start_Year"] = row["Start_Year"]
            weather_records.append(data)

        time.sleep(2)

    weather_df = pd.DataFrame(weather_records)

    df_merged = df_merged.merge(
        weather_df, on=["State", "District", "Season", "Start_Year"], how="left"
    )

    return df_merged

In [5]:
df_final = append_data(df_merged)
df_final

e:\Projects\SBS_HackTheGap_2026\python\sbs\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Fetching soil data in 3 batches...


100%|██████████| 3/3 [10:16<00:00, 205.44s/it]
e:\Projects\SBS_HackTheGap_2026\python\sbs\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Fetching weather data in 96 batches...


100%|██████████| 96/96 [3:37:59<00:00, 136.25s/it]  


,State,District,Season,Year,Crop,Area_Ha,Production_Ton,Yield_TonHa,Start_Year,District_norm,...,longitude,soil_ph,soil_oc,clay_pct,sand_pct,cec_cmol,avg_temp,humidity_avg,rain_total,solar_avg
0,Gujarat,Ahmadabad,Kharif,2015 - 2016,Arhar/Tur,1249.0,1535.0,1.23,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
1,Gujarat,Ahmadabad,Kharif,2015 - 2016,Bajra,755.0,882.0,1.17,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
2,Gujarat,Ahmadabad,Kharif,2015 - 2016,Castor seed,57920.0,86926.0,1.50,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
3,Gujarat,Ahmadabad,Kharif,2015 - 2016,Cotton(lint),131881.0,302016.0,2.29,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,30.040410,69.658033,604.30,18.392459
4,Gujarat,Ahmadabad,Rabi,2015 - 2016,Gram,8087.0,5607.0,0.69,2015,ahmadabad,...,72.299468,7.3,19.0,28.9,32.9,28.5,25.365137,29.049126,1.11,18.429180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75709,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Sweet potato,27.0,313.0,11.59,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033
75710,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Urad,2028.0,1168.0,0.58,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033
75711,Uttar Pradesh,Varanasi,Rabi,2022 - 2023,Wheat,70793.0,173726.0,2.45,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,20.513407,56.130055,122.69,14.440934
75712,Uttar Pradesh,Varanasi,Kharif,2022 - 2023,Sannhamp,377.0,221.0,0.59,2022,varanasi,...,82.936441,7.3,20.9,30.3,30.5,35.4,31.053934,68.999180,748.64,17.463033


In [6]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75714 entries, 0 to 75713
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State           75714 non-null  object 
 1   District        75714 non-null  object 
 2   Season          75714 non-null  object 
 3   Year            75714 non-null  object 
 4   Crop            75714 non-null  object 
 5   Area_Ha         75714 non-null  float64
 6   Production_Ton  75215 non-null  float64
 7   Yield_TonHa     75714 non-null  float64
 8   Start_Year      75714 non-null  int64  
 9   District_norm   75714 non-null  object 
 10  State_norm      75714 non-null  object 
 11  latitude        75714 non-null  float64
 12  longitude       75714 non-null  float64
 13  soil_ph         71885 non-null  float64
 14  soil_oc         71885 non-null  float64
 15  clay_pct        71885 non-null  float64
 16  sand_pct        71885 non-null  float64
 17  cec_cmol        71885 non-null 

In [7]:
df_final.to_csv("data/enriched_crop_data.csv", index=False)